# Stage 5: the coefficient vector as a `Vector`

Expanding a function of the state on a basis gives one coefficient per basis function.
`Vector(expr, basis, u)` is that coefficient vector. `expr` can be an initial distribution or an
observable, and `Vector` treats both in the same way.

Constructing a `Vector` computes no coefficients. `vector[label]` computes the coefficient at that
label, using the same labels as the matrix.

In [1]:
import sympy as sp

from lin_ode.basis import CarlemanBasis, GaussianHermiteBasis, ProjectionError, PSCarlemanBasis
from lin_ode.vector import Vector

u = sp.IndexedBase("u")
initial_value = 3 + u[0] ** 2 + 2 * u[0] * u[1]

basis = CarlemanBasis(n_vars=2, degree=2)
vector = Vector(initial_value, basis, u)

vector[(0, 2)]  # coefficient of u_0^2

1

The basis enumerates its own states in flat order, so one loop over them fills a dense vector.

In [2]:
states = [basis.structured_idx_of_flat_index(k) for k in range(len(basis))]

for k, state in enumerate(states):  # flat index, the monomial it names, its coefficient
    print(f"k={k}  {str(basis[state]):<12} coeff = {vector[state]}")

k=0  CarlemanState() coeff = 3
k=1  CarlemanState((0, 1)) coeff = 0
k=2  CarlemanState((1, 1)) coeff = 0
k=3  CarlemanState((0, 2)) coeff = 1
k=4  CarlemanState((0, 1), (1, 1)) coeff = 2
k=5  CarlemanState((1, 2)) coeff = 0


## Projectors of the three bases

:::{note}
This section and the next are how a coefficient is produced, not something you call. `Vector`
applies the projector for you. Read them when a coefficient is not the number you expected, or
when you need a projector the basis does not ship.
:::

A projector belongs to the basis, not to `Vector`. All three bases ship a projector for
polynomials, and the three compute different coefficients from the same expression.

| basis | projector | the coefficient is |
| --- | --- | --- |
| `CarlemanBasis` | `CarlemanProjector` | the monomial coefficient, a lookup |
| `PSCarlemanBasis` | inherited | the same lookup |
| `GaussianHermiteBasis` | `GHPolyProjector` | $\int\psi_n(v)\,f(v)\,dv$, an integral |

In [3]:
for candidate in (CarlemanBasis(2, 2), PSCarlemanBasis(2, 2, sp.Symbol("s")), GaussianHermiteBasis(2, 2)):
    coefficient = Vector(initial_value, candidate, u)[(0, 2)]
    print(f"{type(candidate).__name__:22} {str(coefficient):20} = {float(coefficient):.6f}")

CarlemanBasis          1                    = 1.000000
PSCarlemanBasis        1                    = 1.000000
GaussianHermiteBasis   8*sqrt(2)*sqrt(pi)   = 20.053026


Carleman returns `1`, the coefficient of $u_0^{2}$. Gauss–Hermite returns $8\sqrt2\sqrt\pi$, an
integral against the whole expression. Both the constant $3$ and the $u_0^{2}$ term contribute to
it, and the cross term $2u_0u_1$ does not, because $\langle\psi_2,v\rangle=0$.

A polynomial is not in $L^2(\mathbb R^N)$, so $\sum_\alpha c_\alpha\psi_\alpha$ does not reconstruct
it. On Gauss–Hermite these coefficients are not an expansion of $g$. They are the weights that turn
a transported density into an expectation,

$$\langle g\rangle=\int g\,\rho\,du=\sum_\alpha \rho_\alpha\,\langle\psi_\alpha,g\rangle.$$

## Non-polynomial factors and missing projectors

`GHPolyProjector` is polynomial-only, so the natural initial density for this basis, a Gaussian, has
no projector.

In [4]:
gaussian_at_half = sp.exp(-((u[0] - sp.Rational(1, 2)) ** 2) / 2) / sp.sqrt(2 * sp.pi)

try:
    Vector(gaussian_at_half, GaussianHermiteBasis(1, 4), u)[(0, 0)]
except ProjectionError as missing:
    print(f"{type(missing).__name__}: {missing}")

ProjectionError: No projector on GaussianHermiteBasis can handle: sqrt(2)*exp(-1/8)*exp(__v/2)*exp(-__v**2/2)/(2*sqrt(pi))


Supplying a projector is the extension point. A projector is any object with `can_project` and
`project`, passed to the basis.

In [5]:
class HermiteProjector:
    """<psi_n, f> for any integrable f: the L2 inner product on the real line."""

    def can_project(self, factor: sp.Expr, v: sp.Symbol) -> bool:
        return True

    def project(self, factor: sp.Expr, n: int, v: sp.Symbol) -> sp.Expr:
        psi = sp.hermite(n, v) * sp.exp(-v**2 / 2) / sp.sqrt(2**n * sp.factorial(n) * sp.sqrt(sp.pi))
        return sp.integrate(factor * psi, (v, -sp.oo, sp.oo))


hermite = GaussianHermiteBasis(n_vars=1, degree=4, projectors=(HermiteProjector(),))
density = Vector(gaussian_at_half, hermite, u)

for n in range(5):
    # expand_func first: the integral comes back as erf + erfc, which simplify alone keeps
    coefficient = sp.simplify(sp.expand_func(density[(0, n)]))
    print(f"<psi_{n} | rho_0> = {str(coefficient):34} {float(coefficient):.3e}")

<psi_0 | rho_0> = sqrt(2)*exp(-1/16)/(2*pi**(1/4))   4.989e-01


<psi_1 | rho_0> = exp(-1/16)/(4*pi**(1/4))           1.764e-01


<psi_2 | rho_0> = exp(-1/16)/(16*pi**(1/4))          4.410e-02


<psi_3 | rho_0> = sqrt(6)*exp(-1/16)/(192*pi**(1/4)) 9.002e-03


<psi_4 | rho_0> = sqrt(3)*exp(-1/16)/(768*pi**(1/4)) 1.591e-03


The coefficients decay fast because the density is smooth, so truncating them is safe here.

| | polynomial $g$ | Gaussian $\rho_0$ |
| --- | --- | --- |
| `CarlemanBasis` / `PSCarlemanBasis` | shipped | no projector, and not the natural basis for a density |
| `GaussianHermiteBasis` | shipped (`GHPolyProjector`) | supply a projector, as above |

- `projectors=` at construction replaces the shipped set.
- Prepending to `basis.projectors` shadows the shipped set for specific factors instead.
- `sympy.integrate` suits one degree of freedom, and is too slow across several.
- Use a closed form where one exists.

---

[← 4. The matrix](05_matrix.ipynb) · [Notebook index](../README.md) · [Boundary conditions →](07_boundary_conditions.ipynb)